In [1]:
 #============================================================
# CELL 1 — Environment setup (MUST RUN FIRST)
# ============================================================
import subprocess
import sys

def install_dependencies():
    base = ["jiwer", "pandas", "numpy"]
    try:
        import mlx_whisper
        packages = base  # mlx already present
    except ImportError:
        try:
            import torch
            if torch.cuda.is_available():
                packages = base + ["openai-whisper"]
            else:
                packages = base + ["openai-whisper", "mlx-whisper"]
        except ImportError:
            packages = base + ["openai-whisper"]

    subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages + ["--quiet"])
    print("✅ Dependencies installed.")

install_dependencies()


/Users/abey/miniconda3/envs/utmos/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Dependencies installed.


DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:

# ============================================================
# CELL 2 — Imports + engine detection
# ============================================================
import os
import math
import gc
import pandas as pd
import numpy as np
import torch
import jiwer
from jiwer import Compose, ToLowerCase, SubstituteWords, RemovePunctuation, Strip, ReduceToListOfListOfWords

print("Detecting hardware...")

if torch.cuda.is_available():
    import whisper as openai_whisper
    ENGINE     = "cuda"
    MODEL_TIER = "medium"
    print(f"🚀 CUDA detected. Loading Whisper '{MODEL_TIER}' on GPU...")
    whisper_model = openai_whisper.load_model(MODEL_TIER, device="cuda")
    print("✅ CUDA engine ready.")
else:
    try:
        import mlx_whisper
        ENGINE     = "mlx"
        MODEL_TIER = "mlx-community/whisper-medium-mlx"
        print("💻 Apple Silicon detected. Using MLX Whisper.")
        print("✅ MLX engine ready.")
    except ImportError:
        import whisper as openai_whisper
        ENGINE     = "cpu"
        MODEL_TIER = "base"   # medium crashes on CPU with word_timestamps — use base
        print(f"⚠️  CPU fallback. Loading Whisper '{MODEL_TIER}'...")
        print("⚠️  Note: using 'base' model on CPU — reduce quality expected.")
        whisper_model = openai_whisper.load_model(MODEL_TIER, device="cpu")
        print("✅ CPU engine ready.")

print(f"✅ All imports done. Engine: {ENGINE}")



Detecting hardware...
💻 Apple Silicon detected. Using MLX Whisper.
✅ MLX engine ready.
✅ All imports done. Engine: mlx


In [3]:

# ============================================================
# CELL 3 — Text normalisation
# ============================================================
# Only Whisper-specific artifacts are normalised.
# "alright" → "all right" : Whisper always splits this regardless of source
# "ok"      → "okay"      : Whisper consistently expands this
# Contractions like gonna/wanna are NOT normalised — they are real deviations
# that should appear in Substitution Detail for the LLM to interpret.

_TRANSFORM = Compose([
    ToLowerCase(),
    SubstituteWords({
        "alright" : "all right",
        "ok"      : "okay",
    }),
    RemovePunctuation(),
    Strip(),
])

def normalize_text(text):
    return _TRANSFORM(text)

print("✅ Normalisation defined.")

✅ Normalisation defined.


In [4]:

# ============================================================
# CELL 4 — Transcription function
# ============================================================
def transcribe(audio_path):
    """
    Transcribes audio and returns per-word probability data.
    Routes to CUDA / MLX / CPU depending on ENGINE set in Cell 2.
    word_timestamps=True enables per-word probability extraction.
    """
    if ENGINE == "cuda":
        result = whisper_model.transcribe(
            audio_path,
            language="en",
            word_timestamps=True,
            fp16=True
        )
    elif ENGINE == "mlx":
        result = mlx_whisper.transcribe(
            audio_path,
            path_or_hf_repo=MODEL_TIER,
            language="en",
            word_timestamps=True
        )
    else:  # cpu
        result = whisper_model.transcribe(
            audio_path,
            language="en",
            word_timestamps=True,
            fp16=False
        )
    return result

print("✅ Transcription function defined.")


✅ Transcription function defined.


In [5]:


# ============================================================
# CELL 5 — WER computation with word-level detail
# ============================================================
def compute_wer(reference_text, hypothesis_text):
    """
    Computes WER and extracts word-level substitution, deletion,
    and insertion detail from jiwer alignment output.
    """
    ref_norm = normalize_text(reference_text)
    hyp_norm = normalize_text(hypothesis_text)

    if not ref_norm or not hyp_norm:
        return {
            "WER"                 : 1.0 if ref_norm else 0.0,
            "Substitutions"       : 0,
            "Deletions"           : 0,
            "Insertions"          : 0,
            "Hits"                : 0,
            "Substitution_Detail" : "—",
            "Deleted_Words"       : "—",
            "Inserted_Words"      : "—",
        }

    out = jiwer.process_words(
        reference_text,
        hypothesis_text,
        reference_transform=Compose([
            ToLowerCase(),
            SubstituteWords({"alright": "all right", "ok": "okay"}),
            RemovePunctuation(),
            Strip(),
            ReduceToListOfListOfWords()
        ]),
        hypothesis_transform=Compose([
            ToLowerCase(),
            SubstituteWords({"alright": "all right", "ok": "okay"}),
            RemovePunctuation(),
            Strip(),
            ReduceToListOfListOfWords()
        ])
    )

    ref_words = ref_norm.split()
    hyp_words = hyp_norm.split()

    substitution_pairs = []
    deleted_words      = []
    inserted_words     = []

    for chunk in out.alignments[0]:
        if chunk.type == "substitute":
            ref_w = ref_words[chunk.ref_start_idx] if chunk.ref_start_idx < len(ref_words) else "?"
            hyp_w = hyp_words[chunk.hyp_start_idx] if chunk.hyp_start_idx < len(hyp_words) else "?"
            substitution_pairs.append(f"{ref_w}→{hyp_w}")
        elif chunk.type == "delete":
            if chunk.ref_start_idx < len(ref_words):
                deleted_words.append(ref_words[chunk.ref_start_idx])
        elif chunk.type == "insert":
            if chunk.hyp_start_idx < len(hyp_words):
                inserted_words.append(hyp_words[chunk.hyp_start_idx])

    return {
        "WER"                 : round(out.wer, 4),
        "Substitutions"       : out.substitutions,
        "Deletions"           : out.deletions,
        "Insertions"          : out.insertions,
        "Hits"                : out.hits,
        "Substitution_Detail" : ", ".join(substitution_pairs) if substitution_pairs else "—",
        "Deleted_Words"       : ", ".join(deleted_words)      if deleted_words      else "—",
        "Inserted_Words"      : ", ".join(inserted_words)     if inserted_words     else "—",
    }

print("✅ WER function defined.")

✅ WER function defined.


In [6]:
#============================================================
# CELL 6 — Intelligibility extraction from Whisper output
# ============================================================
def extract_intelligibility(result, mumble_threshold):
    """
    Extracts per-word log probabilities from Whisper result.
    Whisper returns probability in LINEAR space (0.0–1.0).
    Converted to LOG space: 0.0 = perfectly confident, more negative = less confident.
    Words below mumble_threshold are flagged as low confidence.
    """
    word_log_probs = []
    low_conf_words = []

    for segment in result.get("segments", []):
        for word_data in segment.get("words", []):
            prob     = word_data.get("probability", 1.0)
            word_txt = word_data.get("word", "").strip()
            log_prob = math.log(prob) if prob > 0 else -10.0
            word_log_probs.append(log_prob)
            if log_prob < mumble_threshold:
                low_conf_words.append(f"{word_txt}({round(log_prob, 2)})")

    total           = len(word_log_probs)
    passed          = sum(1 for lp in word_log_probs if lp >= mumble_threshold)
    mean_log_prob   = round(float(np.mean(word_log_probs)), 4) if word_log_probs else None
    intel_pass_rate = round(passed / total, 4) if total > 0 else None

    return {
        "Mean_LogProb"    : mean_log_prob,
        "Intel_Pass_Rate" : intel_pass_rate,
        "Low_Conf_Words"  : ", ".join(low_conf_words) if low_conf_words else "—",
    }

print("✅ Intelligibility extraction defined.")


✅ Intelligibility extraction defined.


In [7]:

# ============================================================
# CELL 7 — Thresholds
# ============================================================
WER_THRESHOLD    = 0.10   # max acceptable word error rate
INTEL_THRESHOLD  = 0.85   # min fraction of words above mumble threshold
MUMBLE_THRESHOLD = -1.0   # log prob below this = low confidence word

print(f"✅ Thresholds set.")
print(f"   WER threshold         : <= {WER_THRESHOLD}")
print(f"   Intelligibility       : >= {INTEL_THRESHOLD} of words clearly spoken")
print(f"   Mumble log prob limit : < {MUMBLE_THRESHOLD}")


✅ Thresholds set.
   WER threshold         : <= 0.1
   Intelligibility       : >= 0.85 of words clearly spoken
   Mumble log prob limit : < -1.0


In [8]:

# ============================================================
# CELL 8 — Configure paths and validate structure
# ============================================================
# ⚠️  UPDATE THESE PATHS FOR YOUR ENVIRONMENT
# Expected structure:
#
# BASE_DIR/
#   ├── references/
#   │   ├── sample1.txt
#   │   └── sample2.txt
#   └── models/
#       ├── model1/
#       │   ├── sample1.wav
#       │   └── sample2.wav
#       └── model2/
#           ├── sample1.wav
#           └── sample2.wav

BASE_DIR       = os.environ.get("WER_BASE_DIR", "/Users/abey/Documents/WER_PER_PRODUCTION/WER_TEST")
REFERENCES_DIR = os.path.join(BASE_DIR, "references")
MODELS_DIR     = os.path.join(BASE_DIR, "models")

for folder in [BASE_DIR, REFERENCES_DIR, MODELS_DIR]:
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")
print("✅ Top level folders found.")

model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([f for f in os.listdir(model_path) if f.endswith(".wav")])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames.")

sample_names = model_samples[model_folders[0]]
for wav_file in sample_names:
    txt_name = os.path.splitext(wav_file)[0] + ".txt"
    txt_path = os.path.join(REFERENCES_DIR, txt_name)
    if not os.path.exists(txt_path):
        raise FileNotFoundError(
            f"Missing reference for {wav_file} — expected: {txt_path}"
        )
print("✅ All reference txt files found.")
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {len(model_folders) * len(sample_names)} evaluations.")



✅ Top level folders found.
✅ Models found: ['model_1', 'model_2']
   model_1: 3 samples
   model_2: 3 samples
✅ All models have identical filenames.
✅ All reference txt files found.

Ready: 2 models × 3 samples = 6 evaluations.


In [9]:

# ============================================================
# CELL 9 — Run evaluation
# ============================================================
import time

results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        audio_path  = os.path.join(MODELS_DIR, model, wav_file)
        txt_path    = os.path.join(REFERENCES_DIR, sample_name + ".txt")

        with open(txt_path, "r", encoding="utf-8") as f:
            reference_text = f.read().strip()

        print(f"\n  Sample : {sample_name}")
        print(f"  Ref    : {reference_text}")

        try:
            result          = transcribe(audio_path)
            hypothesis_text = result["text"].strip()
            print(f"  Whisper: {hypothesis_text}")

            wer_data   = compute_wer(reference_text, hypothesis_text)
            intel_data = extract_intelligibility(result, MUMBLE_THRESHOLD)

            wer_pass   = wer_data["WER"] <= WER_THRESHOLD
            intel_pass = (
                intel_data["Intel_Pass_Rate"] is not None and
                intel_data["Intel_Pass_Rate"] >= INTEL_THRESHOLD
            )
            deletion_flag = wer_data["Deletions"] > 0

            print(f"  WER    : {wer_data['WER']} {'✅' if wer_pass else '❌'} | "
                  f"Intel: {intel_data['Intel_Pass_Rate']} {'✅' if intel_pass else '❌'}")
            if wer_data["Substitution_Detail"] != "—":
                print(f"  Subs   : {wer_data['Substitution_Detail']}")
            if wer_data["Deleted_Words"] != "—":
                print(f"  Deleted: {wer_data['Deleted_Words']}")
            if wer_data["Inserted_Words"] != "—":
                print(f"  Inserted: {wer_data['Inserted_Words']}")
            if intel_data["Low_Conf_Words"] != "—":
                print(f"  Low Conf: {intel_data['Low_Conf_Words']}")

            results.append({
                "Model"               : model,
                "Sample"              : sample_name,
                "Reference"           : reference_text,
                "Whisper"             : hypothesis_text,
                "WER"                 : wer_data["WER"],
                "Substitutions"       : wer_data["Substitutions"],
                "Deletions"           : wer_data["Deletions"],
                "Insertions"          : wer_data["Insertions"],
                "Hits"                : wer_data["Hits"],
                "Substitution_Detail" : wer_data["Substitution_Detail"],
                "Deleted_Words"       : wer_data["Deleted_Words"],
                "Inserted_Words"      : wer_data["Inserted_Words"],
                "Mean_LogProb"        : intel_data["Mean_LogProb"],
                "Intel_Pass_Rate"     : intel_data["Intel_Pass_Rate"],
                "Low_Conf_Words"      : intel_data["Low_Conf_Words"],
                "WER_Pass"            : "✅" if wer_pass   else "❌",
                "Intel_Pass"          : "✅" if intel_pass else "❌",
                "Deletion_Flag"       : "⚠️" if deletion_flag else "✅",
            })

        except Exception as e:
            print(f"  🚨 ERROR: {e}")
            results.append({
                "Model"               : model,
                "Sample"              : sample_name,
                "Reference"           : reference_text,
                "Whisper"             : "ERROR",
                "WER"                 : 1.0,
                "Substitutions"       : 0,
                "Deletions"           : 0,
                "Insertions"          : 0,
                "Hits"                : 0,
                "Substitution_Detail" : "—",
                "Deleted_Words"       : "—",
                "Inserted_Words"      : "—",
                "Mean_LogProb"        : None,
                "Intel_Pass_Rate"     : None,
                "Low_Conf_Words"      : "ERROR",
                "WER_Pass"            : "❌",
                "Intel_Pass"          : "❌",
                "Deletion_Flag"       : "—",
            })

        finally:
            if 'result' in locals():
                del result
            gc.collect()
            if ENGINE == "mlx":
                time.sleep(0.5)  # MLX thread settle — not needed on CUDA/CPU

print("\n\nAll evaluations complete.")




Model: model_1

  Sample : suits_base_sample copy 2
  Ref    : But don't take too long, alright? And if you lay a hand on me one more time I'll have you thrown out of the bar faster than your disgraced mentor is.


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 56299.38it/s]


  Whisper: But don't take too long, all right? And if you lay a hand on me one more time, I'll have you thrown out of the bar faster than your disgraced mentor.
  WER    : 0.0312 ✅ | Intel: 1.0 ✅
  Deleted: is

  Sample : suits_base_sample copy
  Ref    : But don't take too long, alright? And if you lay a hand on me one more time I'll have you thrown out of the bar faster than your disgraced.
  Whisper: But don't take too long, all right? And if you lay a hand on me one more time, I'll have you thrown out of the bar faster than your disgraced mentor.
  WER    : 0.0333 ✅ | Intel: 1.0 ✅
  Inserted: mentor

  Sample : suits_base_sample
  Ref    : But don't take too long, alright? And if you lay a hand on me one more time I'll have you thrown out of the bar faster than your disgraced mentor.
  Whisper: But don't take too long, all right? And if you lay a hand on me one more time, I'll have you thrown out of the bar faster than your disgraced mentor.
  WER    : 0.0 ✅ | Intel: 1.0 ✅

Model: 

In [10]:

# ============================================================
# CELL 10 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)

df["Both_Pass"] = df.apply(
    lambda row: "✅" if (row["WER_Pass"] == "✅" and row["Intel_Pass"] == "✅") else "❌",
    axis=1
)

# ── Table 1 — full per-segment results ──
print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[[
    "Model", "Sample", "Reference", "Whisper",
    "WER", "Substitutions", "Deletions", "Insertions",
    "Substitution_Detail", "Deleted_Words", "Inserted_Words",
    "Mean_LogProb", "Intel_Pass_Rate", "Low_Conf_Words",
    "WER_Pass", "Intel_Pass", "Both_Pass", "Deletion_Flag"
]].to_string(index=False))

# ── Table 2 — per-model summary ──
print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df     = df[df["Model"] == model]
    total        = len(model_df)
    wer_vals     = model_df["WER"]
    logprob_vals = model_df["Mean_LogProb"].dropna()

    wer_pass_count   = (model_df["WER_Pass"]   == "✅").sum()
    intel_pass_count = (model_df["Intel_Pass"] == "✅").sum()
    both_pass_count  = (model_df["Both_Pass"]  == "✅").sum()

    summary_rows.append({
        "Model"              : model,
        "Segments"           : total,
        "Both Pass Rate"     : f"{both_pass_count}/{total}",
        "WER Pass Rate"      : f"{wer_pass_count}/{total}",
        "Intel Pass Rate"    : f"{intel_pass_count}/{total}",
        "Median WER"         : round(wer_vals.median(), 4),
        "Max WER"            : round(wer_vals.max(), 4),
        "Median LogProb"     : round(logprob_vals.median(), 4) if len(logprob_vals) > 0 else None,
        "Total Deletions"    : model_df["Deletions"].sum(),
        "Total Substitutions": model_df["Substitutions"].sum(),
        "Total Insertions"   : model_df["Insertions"].sum(),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Table 3 — model ranking ──
print("\n========== MODEL RANKING ==========")
print("Primary   → Both Pass Rate — must pass WER and Intelligibility")
print("Tiebreak1 → WER Pass Rate (highest first)")
print("Tiebreak2 → Intel Pass Rate (highest first)")
print("Tiebreak3 → Total Deletions (lowest first)")
print("Tiebreak4 → Median WER (lowest first)")
print("Tiebreak5 → Max WER (lowest first)\n")

summary_df["_both_pass_num"]  = summary_df["Both Pass Rate"].apply(lambda x: int(x.split("/")[0]))
summary_df["_wer_pass_num"]   = summary_df["WER Pass Rate"].apply(lambda x: int(x.split("/")[0]))
summary_df["_intel_pass_num"] = summary_df["Intel Pass Rate"].apply(lambda x: int(x.split("/")[0]))

ranking = summary_df.sort_values(
    by=["_both_pass_num", "_wer_pass_num", "_intel_pass_num", "Total Deletions", "Median WER", "Max WER"],
    ascending=[False, False, False, True, True, True]
)[[
    "Model", "Both Pass Rate", "WER Pass Rate", "Intel Pass Rate",
    "Median WER", "Max WER", "Median LogProb",
    "Total Deletions", "Total Substitutions"
]]

print(ranking.to_string(index=False))

print("\n========== WHAT TO LOOK FOR ==========")
print("Both Pass Rate     → primary ranking — segment passed WER AND intelligibility")
print("WER Pass Rate      → % of segments with acceptable word accuracy")
print("Intel Pass Rate    → % of segments with clearly spoken words")
print("Median WER         → typical error level — large gap vs Max = unpredictable model")
print("Max WER            → worst segment performance")
print("Median LogProb     → typical Whisper confidence across segments")
print("Total Deletions    → words completely dropped — safety critical for dialogue")
print("Substitution Detail→ exact word pairs swapped — check colloquial vs real error")
print("Low Conf Words     → specific words Whisper struggled with — listen manually")
print("Deletion Flag      → any segment with dropped words — always review")
print(f"\n[Engine used: {ENGINE} | Model: {MODEL_TIER}]")


========== FULL PER-SEGMENT RESULTS ==========
  Model                   Sample                                                                                                                                             Reference                                                                                                                                               Whisper    WER  Substitutions  Deletions  Insertions Substitution_Detail Deleted_Words Inserted_Words  Mean_LogProb  Intel_Pass_Rate Low_Conf_Words WER_Pass Intel_Pass Both_Pass Deletion_Flag
model_1 suits_base_sample copy 2 But don't take too long, alright? And if you lay a hand on me one more time I'll have you thrown out of the bar faster than your disgraced mentor is. But don't take too long, all right? And if you lay a hand on me one more time, I'll have you thrown out of the bar faster than your disgraced mentor. 0.0312              0          1           0                   —            is              —        -

In [12]:
df = pd.DataFrame(results)

In [13]:
# final cell in each gate notebook
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)
print("✅ Results saved to results.csv")

✅ Results saved to results.csv
